# Welcome to the MLOps Workshop: From Notebook to Production

Welcome! You are about to embark on a hands-on journey through **MLOps** -- the discipline of putting machine learning models into production and keeping them running reliably.

**By the end of this workshop series, you will understand the full ML lifecycle in production.** You will not just read about it -- you will build every piece yourself, from data pipelines to monitoring dashboards.

### What You'll Learn in This Notebook

- Why deploying ML models is surprisingly hard
- What MLOps is and why it exists
- The full ML lifecycle, from data to monitoring
- The tools we will use throughout this workshop
- A first look at our project: **Energy Demand Forecasting**

### Prerequisites

You should be comfortable with:
- **Python** (functions, classes, packages)
- **Basic ML concepts** (what a model is, training vs. testing, what overfitting means)

That's it. No prior experience with deployment, DevOps, Docker, or any of the tools we will use is required. We will explain everything from scratch.

---
## The Problem: Why ML in Production is Hard

Let's tell a story.

> You spent two weeks building a model that predicts hourly energy demand for office buildings. It works great on your test set -- your RMSE is low, your plots look beautiful, and you are proud of it.
>
> Your manager walks over and says: **"Great work! Ship it to production by Friday."**
>
> You nod confidently. Then you sit down and realize you have no idea what "ship it to production" actually means.

Here are the questions that start flooding in:

1. **How do I serve predictions?** My model lives in a Jupyter notebook. The operations team needs an API they can call. How do I turn `model.predict(X)` into something a web application can use?

2. **How do I retrain it?** Energy patterns change with the seasons. The model trained on summer data will be terrible in winter. Who retrains it? How often? Manually?

3. **How do I know if it is still working?** The model might silently start giving bad predictions. No one will notice until a building wastes thousands of dollars on energy. How do I detect this?

4. **How do I track what I tried?** You tried 47 different feature combinations, 3 model architectures, and 12 hyperparameter settings. Which one did you deploy? Can you reproduce it?

5. **How do I handle bad data?** What happens when a sensor breaks and starts sending zeros? Or when someone changes the data format upstream?

6. **How do I test changes safely?** You want to try a new model. How do you test it against the current one without breaking anything?

These are not ML problems. They are **engineering** problems. And they are the reason that, according to various industry surveys, **most ML projects never make it to production.**

This is the gap that MLOps fills.

---
## What is MLOps?

**MLOps = Machine Learning + Operations**

It is the set of practices, tools, and culture for **deploying and maintaining ML models in production reliably and efficiently.**

If you have heard of **DevOps**, MLOps is the same idea applied to machine learning:

| | DevOps | MLOps |
|---|---|---|
| **Goal** | Ship reliable software | Ship reliable ML systems |
| **Artifacts** | Code, binaries | Code, data, models |
| **Testing** | Unit tests, integration tests | + data tests, model tests |
| **Deployment** | Deploy code | Deploy code + models |
| **Monitoring** | Latency, errors, uptime | + model accuracy, data drift |

### The Key Difference

In traditional software, code is the main thing you manage. In ML systems, **data is a first-class citizen**. Your model is only as good as the data it was trained on, and that data changes over time.

This is why you cannot just use DevOps for ML. You need additional practices for:
- **Data versioning** -- tracking which data a model was trained on
- **Experiment tracking** -- recording every training run and its results
- **Model registry** -- storing and versioning trained models
- **Data/model monitoring** -- detecting when data or model quality degrades
- **Automated retraining** -- updating models when they go stale

MLOps is not a single tool. It is a **set of practices** that you adopt incrementally. You do not need everything on day one.

---
## The ML Lifecycle

ML in production is not a straight line from data to model. It is a **cycle**. Here is the full picture:

```
    +---------------------+
    |                     |
    v                     |
+---+---+   +----------+---+----------+   +-----------+
| 1. Data|-->| 2. Feature  |-->| 3. Model   |-->| 4. Model  |
| Collect|   | Engineering |   | Training & |   | Evaluation|
| & Valid|   |             |   | Experiment |   | & Valid.  |
+--------+   +-------------+   +------------+   +-----+-----+
                                                       |
                                                       v
+------------+   +-----------+                  +------+------+
| 6. Monitor |<--| 5. Deploy |<-----------------| Pass?       |
| & Feedback |   | & Serve   |                  | Ship it!    |
+-----+------+   +-----------+                  +-------------+
      |
      +--- (loop back to step 1 when things change) --->
```

Let's walk through each step using our energy forecasting project as an example:

### 1. Data Collection & Validation
We collect hourly energy readings from smart meters in buildings, along with weather data and building metadata. Before using this data, we **validate** it: Are there missing readings? Are the values within reasonable ranges? Has the format changed since last time?

### 2. Feature Engineering
Raw timestamps and meter readings are not enough. We create features like "hour of day," "day of week," "is it a holiday," "rolling average temperature over 24 hours," and so on. These transformations must be **reproducible** -- the same logic applied during training must be applied during prediction.

### 3. Model Training & Experimentation
We try different algorithms (linear regression, gradient boosting, neural networks), different hyperparameters, and different feature sets. Each attempt is an **experiment** that we track so we can compare results and reproduce the best one.

### 4. Model Evaluation & Validation
We evaluate the best model on held-out test data. But we go beyond just accuracy metrics -- we check for fairness across building types, performance on edge cases (holidays, extreme weather), and whether the model is actually better than the one currently in production.

### 5. Model Deployment & Serving
The approved model gets packaged and deployed behind an API. Operations teams and applications can now call this API to get energy demand predictions. We need to think about latency, throughput, and graceful failure.

### 6. Monitoring & Feedback
Once the model is live, we monitor it continuously. Is prediction accuracy holding up? Has the input data distribution shifted (maybe a new building type was added)? Are there any errors? When things degrade, we trigger a retraining cycle -- and we are back to step 1.

**The crucial insight: this is a CYCLE, not a one-time project.** A model that is deployed and forgotten will eventually fail. MLOps is about making this cycle fast, reliable, and as automated as possible.

---
## MLOps Maturity Levels

You do not go from zero to full automation overnight. The industry commonly describes four maturity levels:

### Level 0: Manual Everything
- A data scientist trains a model in a Jupyter notebook
- Someone manually copies the model file to a server
- No experiment tracking, no monitoring, no automated retraining
- **This is where most teams start** -- and many stay here

### Level 1: ML Pipeline Automation
- Training is automated: a pipeline handles data loading, feature engineering, training, and evaluation
- Experiment tracking is in place (you know what was trained and how)
- Deployment is still manual: someone reviews results and deploys by hand

### Level 2: CI/CD Pipeline Automation
- Training AND deployment are automated
- Automated tests verify data quality, model quality, and code quality before deployment
- A model that passes all tests gets deployed automatically

### Level 3: Full Automation with Monitoring & Retraining
- Everything in Level 2, plus:
- Continuous monitoring detects data drift and model degradation
- Retraining is triggered automatically when performance drops
- The system is self-healing

**In this workshop, we will build a Level 2-3 system.** We will start simple and add sophistication incrementally, just like you would in a real project.

---
## Our Project: Energy Demand Forecasting

Throughout this workshop, we will build a complete MLOps system for **predicting hourly energy demand in buildings.**

### Why Energy Forecasting?

This is a great use case for learning MLOps because it has all the challenges you will face in real-world ML:

- **Data changes over time.** Energy patterns shift with seasons, occupancy changes, and building renovations. A model trained in July will struggle in January. This is called **data drift**, and it is one of the most common reasons ML models fail in production.

- **Predictions have real consequences.** Bad forecasts lead to wasted energy and money. We need monitoring to catch problems quickly.

- **New data arrives continuously.** Buildings generate readings every hour. We need pipelines that process this data reliably.

- **Multiple models are needed.** Different building types may need different models. We need a way to manage them all.

### What We'll Build

By the end of this workshop, you will have:

- A **data pipeline** that generates, validates, and transforms energy data
- An **experiment tracking system** that records every training run
- Multiple **trained models** with proper evaluation
- An **automated training pipeline** that retrains on schedule
- A **REST API** that serves predictions
- A **monitoring dashboard** that tracks model and data health
- **CI/CD pipelines** that test and deploy automatically
- An **orchestration system** that ties everything together

Let's get a first look at the data we will be working with.

In [ ]:
# Let's see what we're building -- generate some energy data
import sys
sys.path.insert(0, '../src')

from energy_forecast.data.synthetic import SyntheticDataGenerator
import matplotlib.pyplot as plt

gen = SyntheticDataGenerator(
    num_buildings=3,
    start_date="2023-01-01",
    end_date="2023-03-31",
    random_seed=42,
)
df = gen.generate()

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print()
df.head(10)

In [ ]:
# Visualize energy demand patterns for one building
building = df[df['building_id'] == df['building_id'].unique()[0]]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Daily pattern (1 week)
week = building.head(168)  # 7 days * 24 hours
axes[0].plot(week['timestamp'], week['energy_demand_kwh'])
axes[0].set_title('Energy Demand - One Week (Daily Pattern)')
axes[0].set_ylabel('kWh')
axes[0].grid(True, alpha=0.3)

# Weekly pattern (1 month)
month = building.head(720)  # ~30 days * 24 hours
axes[1].plot(month['timestamp'], month['energy_demand_kwh'])
axes[1].set_title('Energy Demand - One Month (Weekly Pattern)')
axes[1].set_ylabel('kWh')
axes[1].grid(True, alpha=0.3)

# Full period
axes[2].plot(building['timestamp'], building['energy_demand_kwh'], alpha=0.7)
axes[2].set_title('Energy Demand - Full Period (Seasonal Trend)')
axes[2].set_ylabel('kWh')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice the patterns: daily peaks during work hours, lower demand on weekends,")
print("and a seasonal trend across months. These are the patterns our models will learn.")

---
## The Tools We'll Use

Each tool in our stack solves a specific MLOps problem. You do not need to know any of these yet -- we will introduce each one when we need it.

| Tool | MLOps Problem It Solves |
|------|------------------------|
| **Pandas / NumPy** | Data processing and feature engineering |
| **scikit-learn / XGBoost / PyTorch** | Model training and evaluation |
| **MLflow** | Experiment tracking and model registry |
| **FastAPI** | Model serving (turning models into APIs) |
| **Prometheus + Grafana** | Monitoring model and system health |
| **Docker** | Reproducible environments ("works on my machine" problem) |
| **Airflow** | Pipeline orchestration (scheduling and dependencies) |
| **GitHub Actions** | CI/CD (automated testing and deployment) |
| **pytest** | Testing code, data, and models |

Don't worry if this looks like a lot. We will introduce each tool one at a time, and you will see exactly why it is needed before we use it.

---
## Workshop Roadmap

Here is where we are headed. Each notebook builds on the previous one:

| Notebook | Topic | What You'll Build |
|----------|-------|-------------------|
| **00 (this one)** | Introduction to MLOps | Understanding of the landscape |
| **01** | Data Engineering | Reliable data pipelines with validation |
| **02** | Experiment Tracking | MLflow setup -- never lose track of what you tried |
| **03** | Model Development | From baseline to advanced models |
| **04** | Training Pipelines | Automated, reproducible training |
| **05** | Model Serving | REST API for predictions with FastAPI |
| **06** | Monitoring | Dashboards for data drift and model health |
| **07** | CI/CD | Automated quality gates with GitHub Actions |
| **08** | Orchestration | Scheduling everything with Airflow |
| **09** | Putting It All Together | The full end-to-end system |

Each notebook is designed to be completed in one sitting (1-2 hours). By the end, you will have a production-grade MLOps system that you built with your own hands.

---
## Key Takeaways

Before we move on, let's recap what we have learned:

1. **Training a model is only a small part of the work.** The real challenge is deploying it, keeping it running, and knowing when it breaks.

2. **MLOps bridges the gap between experimentation and production.** It is a set of practices -- not a single tool -- for making ML systems reliable, reproducible, and automated.

3. **ML systems are different from traditional software** because data is a first-class citizen. Data changes, models degrade, and the system must adapt.

4. **You adopt MLOps incrementally.** Start at Level 0 and work your way up. Every step adds value.

5. **We will build everything hands-on** in this workshop, using energy demand forecasting as our real-world project.

Ready? Let's make sure your environment is set up, and then we will move on to **Notebook 01: Data Engineering**.

In [ ]:
# Quick environment check -- let's verify all required packages are installed
import importlib

print("Checking your environment...")
print("=" * 40)

packages = [
    'numpy',
    'pandas',
    'sklearn',
    'xgboost',
    'torch',
    'mlflow',
    'fastapi',
    'prometheus_client',
]

all_ok = True
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'installed')
        print(f"  {pkg}: {ver}")
    except ImportError:
        print(f"  {pkg}: NOT INSTALLED - run: pip install -e '.[dev,test]'")
        all_ok = False

print("=" * 40)
if all_ok:
    print("\nAll packages found. You are ready for the workshop!")
else:
    print("\nSome packages are missing. Install them before continuing.")
    print("From the project root, run: pip install -e '.[dev,test]'")